# NB-05 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-05**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-05.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This tests what a spectral cut measures and why a structural bottleneck alone does not establish economic congestion.

- [Volume II, Chapter 5 — Spectral theory, cuts, and oscillations](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c5)
- [Volume III, §3.1 — A two-region grid, worked out](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c3-w)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-05', require_key=True)
print(setup_status)


# NB-05 — Spectral cuts and what a bottleneck does not prove
Companion to Volume III §§1–3 and Volume II §5. Prerequisite: NB-04's Laplacian.
We reuse the six-bus barbell, its four published bridge weights, the saved
System E graph specification, and retained controlled market observations.
No random graph, new demand sample, or market-year simulation is created.

We compare normalized and combinatorial spectra, enumerate small-graph cuts,
and separate structural weakness from binding transmission constraints.

In [ ]:
NOTEBOOK_ID = 'NB-05'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

manifest=inputs['manifest'];a=np.asarray(inputs['graphs']['barbell']['adjacency']);e=np.asarray(inputs['graphs']['system_e']['adjacency'])
print('Local declared graph inspection:',inputs['graphs']['barbell'])
operators={name:execute(name,'graph_operators',request)['laplacian_bundle'] for name,request in inputs['requests'].items()}
def cut(name,graph_name,coordinates,parents=()):
    graph=inputs['graphs'][graph_name]
    source_hash=responses[parents[0]]['receipt']['result_sha256'] if parents else session.input_manifest['inputs_sha256']
    request=dict(inputs=dict(undirected_graph=graph,sweep_coordinate=dict(node_ids=graph['node_ids'],values=np.asarray(coordinates).tolist(),coordinate_unit='dimensionless',context=graph['context'],
        source=dict(kind='caller_declared_scalar',reference_id='NB-05:'+name,artifact_sha256=source_hash),threshold_policy='distinct_values',degeneracy_policy='reject_constant')),parameters=dict(score='two_sided'),context=dict(input_origin='Original NB-05 graph; explicit scalar threshold or binary subset; no multiway or market claim'))
    return execute(name,'spectral_cut',request,parents)['graph_partition']
reports={}
for name,graph in inputs['graphs'].items():
    parent=name+':symmetric_normalized';normal=operators[parent];comb=operators[name+':combinatorial']
    coordinate=np.asarray(normal['spectrum']['eigenvectors_columns'])[:,1]/np.sqrt(normal['degrees'])
    swept=cut(name+':sweep',name,coordinate,[parent])
    sign=cut(name+':sign',name,np.where(coordinate>0,0.,1.),[parent])
    reports[name]=dict(combinatorial=np.array(comb['spectrum']['eigenvalues']),normalized=np.array(normal['spectrum']['eigenvalues']),coordinates=coordinate,
        selected=np.array([int(n) for n in swept['selected_node_ids']]),score=swept['selected_score'],sign_score=sign['selected_score'],cuts=[])
for mask in range(31):
    side=[0]+[i+1 for i in range(5) if mask&(1<<i)]
    binary=[0. if i in side else 1. for i in range(6)]
    result=cut('exact:'+str(mask),'barbell',binary)
    assert set(map(int,result['selected_node_ids']))==set(side) and len(result['ledger'])==1
    ledger=result['ledger'][0];reports['barbell']['cuts'].append(dict(side=side,one_sided=ledger['one_sided'],two_sided=ledger['two_sided']))
b=reports['barbell'];system=reports['system_e']
assert len(b['cuts'])==31
np.testing.assert_allclose(b['combinatorial'],manifest['exact_registry']['spectrum'],atol=5e-4)
print('Barbell combinatorial spectrum:',b['combinatorial'].round(6))
print('Barbell normalized spectrum:',b['normalized'].round(6))
print('System E (rounded saved specification) normalized λ₂:',system['normalized'][1])

## Degree normalization changes the spectral question
L=D−A acts on nodal coordinates; ℒ=D⁻¹ᐟ²LD⁻¹ᐟ² has a dimensionless spectrum.
To sweep a normalized eigenvector v, first use D⁻¹ᐟ²v. Cut boundaries and volumes
are measured in the original nonnegative weighted adjacency. A signed cosine
matrix is not admissible as this graph, even if it is a valid exposure kernel.
The normalized eigenvector's global sign changes cut orientation, not its score.

In [ ]:
plt=setup_plot();fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(range(1,7),b['combinatorial'],'o-',color='#88C0D0');axes[0].set(xlabel='Mode',ylabel='Eigenvalue',title='Combinatorial L · barbell')
axes[1].plot(range(1,7),b['normalized'],'o-',color='#B48EAD');axes[1].set(xlabel='Mode',ylabel='Eigenvalue',title='Normalized ℒ · same graph')
fig.tight_layout();plt.show()
degree=np.asarray(operators['barbell:symmetric_normalized']['degrees'])
print('Sweep coordinate:',b['coordinates'].round(6))
reverse_result=cut('barbell:reverse','barbell',-b['coordinates'],['barbell:symmetric_normalized']);reverse_score=reverse_result['selected_score'];reverse=np.array([int(n) for n in reverse_result['selected_node_ids']])
np.testing.assert_allclose(reverse_score,b['score'],atol=1e-12)

## Exhaust all 31 distinct bipartitions
One-sided φ(S)=boundary/volume(S). The two-sided score is
max(φ(S),φ(complement)) = boundary/min(volume(S),volume(complement)).
The sign split, threshold sweep and exact best bipartition agree for this barbell;
that fixture agreement is not a generic optimality guarantee for sign splits.

In [ ]:
ordered=sorted(b['cuts'],key=lambda r:(r['two_sided'],r['side']))
for row in ordered[:5]:print(row)
best=ordered[0]['two_sided']
np.testing.assert_allclose([best,b['sign_score']],[b['score']]*2,atol=1e-12)
assert b['normalized'][1]/2<=best+1e-12<=np.sqrt(2*b['normalized'][1])+1e-12
print('Sweep buses (published labels):',(b['selected']+1).tolist(),'two-sided score:',b['score'])
fig,ax=plt.subplots(figsize=(10,4));ax.plot(range(1,32),[r['two_sided'] for r in ordered],'o',color='#88C0D0')
ax.axhline(b['score'],color='#A3BE8C',label='Sweep = exact best in this fixture')
ax.set(xlabel='Bipartition rank',ylabel='Two-sided conductance',title='All barbell cuts, measured on the original graph');ax.legend();fig.tight_layout();plt.show()

## Reuse the published bridge-weight sensitivity
Only the four bridge weights from the Model D manifest are used. Susceptance
weights alter L and ℒ; thermal capacity is a separate model input. No line
capacity is inferred from a conductance score or from a small eigenvalue.

In [ ]:
scaling=[]
for weight,want in manifest['exact_registry']['lambda2_scaling'].items():
    changed=a.copy();changed[2,3]=changed[3,2]=float(weight)
    r=reports['bridge:'+str(weight)]
    assert abs(r['combinatorial'][1]-want)<5e-4
    scaling.append({'bridge_weight':float(weight),'lambda2':float(r['combinatorial'][1]),
                    'normalized_lambda2':float(r['normalized'][1]),'conductance':r['score']})
print(scaling)
reference=inputs['saved_cheeger']
np.testing.assert_allclose(system['score'],reference['sweep_conductance'],atol=1e-9)
np.testing.assert_allclose(system['normalized'][1],reference['normalized_lambda2'],atol=1e-9)
print('System E sweep score:',system['score'],'; prior saved score:',reference['sweep_conductance'])

## Same weak interface, different economic activity
The existing System E market panel contains both a zero-dual hour and a
positive-dual hour for tie A. These retained intact, no-shedding observations
have the same weak weight (0.5) and limit. A weak connection alone does not imply
congestion; injections, limits, and the economic dispatch matter. These are two
controlled illustrations, not a causal estimate or real-market observations.

In [ ]:
examples=inputs['examples']
for row in examples:print(row)
assert examples[0]['limit_MW']==examples[1]['limit_MW']
assert examples[0]['dual']==0 and examples[1]['dual']>0
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot([r['bridge_weight'] for r in scaling],[r['conductance'] for r in scaling],'o-',color='#88C0D0')
axes[0].set(xlabel='Published bridge weight',ylabel='Two-sided conductance',title='Barbell structural sensitivity')
axes[1].bar(range(2),[r['dual'] for r in examples],color=['#88C0D0','#D08770']);axes[1].set_xticks(range(2),['Retained hour 0','Retained hour 60'])
axes[1].set(ylabel='Signed tie-A dual · toy currency/MWh',title='System E · identical weight and limit');fig.tight_layout();plt.show()

## Multiway theorem: what this calculation does not implement
For k disjoint nonempty sets, let ρ(k) minimize their maximum one-sided
conductance; they need not partition all vertices. The manuscript's pinned
[Li preprint, v1](https://arxiv.org/abs/2609.35369v1) states
ρ(k) ≤ C[1+log(k+1)]⁵√λₖ(ℒ), with an absolute constant C.
Its construction yields exactly k sets and measures the cuts in the original
graph. The binary sweep above is not that proof construction, a k-means
guarantee, or an assertion about trading zones. We do not fit C to this fixture.

Exercise: why cannot the normalized eigenvalue be substituted for the
combinatorial value in the published bridge-weight table? Their scaling and
inner products differ. A structural bottleneck still needs dispatch validation.

In [ ]:
for name,adjacency in [('signed adjacency',-a),('zero degree',np.zeros_like(a))]:
    bad=copy.deepcopy(inputs['requests']['barbell:combinatorial']);bad['inputs']['undirected_graph']['adjacency']=adjacency.tolist();reject('graph_operators',bad)

In [ ]:
def plain(v):
    if isinstance(v,np.ndarray):return v.tolist()
    raise TypeError(type(v))
(session.output/'consumer-summary.json').write_text(json.dumps(dict(reports=reports,scaling=scaling,existing_market_examples=examples),default=plain,allow_nan=False,indent=2)+'\n')
summary=session.finish({'complete_31_cuts':True,'declared_operator_spectra':True,'saved_sensitivity_parity':True,'retained_market_examples_only':True,'new_synthetic_data':False})